## Setup

In [12]:
from pathlib import Path

import aw
from aw import reload
from IPython.display import display
import pandas as pd

import job_search.ai as ai
from job_search.config import DS_HEALTH, DS_NORCAL, P_CACHE, VIEW_JOB_HTTPS, P_URLS, P_DICT, P_JOBS
import job_search.config as conf
import job_search.dataset as ds
import job_search.jobs as jobs
import job_search.utils as utils
from job_search.jobs import COLS, cmask, disp

In [2]:
from job_search.config import P_DATA, P_QUERY
# P_save_ = P_DATA / 'processed/2026-02-15/DS_NorCal' / 'DS_NorCal.html'
# P_save = P_DATA / 'processed/2026-02-18/DS_NorCal' / 'DS_NorCal.html'
# P_save = P_DATA / 'processed/2026-02-21/DS_NorCal' / 'DS_NorCal.html'
P_query = P_QUERY / ("DS_NorCal.txt")
# P_save = ds.main0(P_query, overwrite=False, bare=True)
# P_save = P_DATA / 'processed/2026-02-24/DS_NorCal' / 'DS_NorCal.html'
jdf = ds.load_jdf()
# jdf.head(3)
jdf_hash2position = dict(zip(jdf['hash'], jdf['position']))

In [ ]:
jobs_df = jobs.load_jobs()
hash2position = dict(zip(jobs_df['_hash'], jobs_df['position']))

In [4]:
jobm = pd.Series(Path('jobm.log').read_text().split('\n'))
jobm_hashes = jobm[jobm.str.startswith('Skip')].str.rsplit('.', n=3).str[1]
jobm_hashes

3                  Z3JuaHNlX19fYTE2el9fXzYyNjMwNjIwMDM=
5                                      nlc47kkfgpl93nb9
7                                      hdivhr68bos5blqb
8                                      515tm1mhy2sggymg
11                                     d08htgjfszw4iguw
13    d29ya2RheV9fX2NhcmRpbmFsaGVhbHRoLXdkMS1leHRfX1...
15                                     jffe2zjyi97kqakr
16                                     qrjydn6q3z51cdot
17                                     qp8g2rk4h8mr1rj1
23                                     whc4tm6h20zb9j0t
24                                     wmkwlb1w9793rzme
29    bGV2ZXJfX19sYW1pbmlhaV9fX2I0MWY4MGExLTA4N2ItND...
35                                     23b7p6u6duzy16gd
38                                     j0dxjmhai8wvlnh9
42                                     t1d1igg1f53susdx
45                                     fg7sespi5mqvu2ic
47    bGV2ZXJfX19hcnRlcmEtMl9fXzhhM2I2NjE2LTY0ODItNG...
48                                     l6l4z5vf1

In [5]:
mdf = pd.merge(jdf[['company', 'hash', 'position']],
               jobs_df[['_hash', 'company_name', 'position']],
               left_on=['hash'], right_on=['_hash'])
mdf.query('position_x != position_y')
mdf.query('position_x != position_y')[['position_x', 'position_y']].style
# mdf.query('position_x != position_y')[['position_x', 'position_y']].shape

,position_x,position_y
1,ASAPP - Lead Machine Learning Engineer,Asapp-2 - Lead Machine Learning Engineer
2,Adobe - Applied ML Research Engineer,- Applied ML Research Engineer
4,Adobe - Sr Machine Learning Engineer,- Sr Machine Learning Engineer
5,"Aircall - Data Engineer, Data and Science","- Data Engineer, Data and Science"
6,"Airwallex - Staff _ Senior Backend Engineer, Agentic AI","- Staff _ Senior Backend Engineer, Agentic AI"
7,American Express - AI Engineer I - Agentic AI,- AI Engineer I - Agentic AI
8,American Express - AI Engineer II - Agentic AI,- AI Engineer II - Agentic AI
9,American Express - AI Engineer III - Agentic AI,- AI Engineer III - Agentic AI
10,American Express - Senior AI Engineer I - Agentic AI,- Senior AI Engineer I - Agentic AI
12,"Anthropic - Data Science Engineer, Capacity and Efficiency","- Data Science Engineer, Capacity and Efficiency"


In [11]:
mdf.query('hash in @jobm_hashes')[['position_x']].style

,position_x
29,"Calico - Senior Data Engineer, Drug Discovery Data Engineering"


In [13]:
ii = 1
(mdf.query('position_x != position_y')['position_x'].iloc[ii],
 mdf.query('position_x != position_y')['position_y'].iloc[ii])

('Adobe - Applied ML Research Engineer', '- Applied ML Research Engineer')

In [8]:
_query_pd_series = jobs._load_query_dict()[DS_NORCAL]
_dfs = [ds.load_jdf(path) for path in _query_pd_series]
query_jdf = pd.concat(_dfs).drop_duplicates(subset='hash', keep='last').reset_index(drop=True)

In [9]:
query_jdf.query('hash == "35a85k2p5oc8hw7j"')

,days,title,location,salary,onsite,full_time,company,company_stock,company_summary,yoe,...,hash,chash,_len,hours,position,_position,lower,upper,median,bay
1932,2w,"Senior Data Scientist , Agents",San Bruno,$164k-$246k/yr,Hybrid,Full Time,Verily,-,: Developing data-driven solutions and tools f...,NaN,...,35a85k2p5oc8hw7j,d29ya2RheV9fX3ZlcmlseS13ZDEtdmVyaWx5X2NhcmVlcn...,0,336,"Verily - Senior Data Scientist , Agents","verily - senior data scientist , agents",164.0,246.0,205.0,"(San Bruno,)"


In [13]:
P_URLS_ = P_URLS.parent / '_urls'
P_DICT_ = P_DICT.parent / '_dicts'
P_JOBS_ = P_JOBS.parent / '_jobs'
P_URLS_.mkdir(exist_ok=True)
P_DICT_.mkdir(exist_ok=True)
P_JOBS_.mkdir(exist_ok=True)
assert P_URLS_.exists() and P_DICT_.exists() and P_JOBS_.exists()

In [14]:
jobm_df = query_jdf.query('hash in @jobm_hashes').drop_duplicates(subset='hash', keep='last').reset_index(drop=True)
jobm_df.shape

(18, 23)

In [15]:
# query_jdf.query('hash in @jobm_hashes').drop_duplicates(subset='hash', keep='last')[['hash']].pipe(aw.disp100)
MOVE = True

for ii, _jobm_hash in enumerate(jobm_df['hash']):
    print(f"{ii:2}: {_jobm_hash}")
    path_dict = {
        P_URLS.name: [p for p in P_URLS.rglob(f'*.{_jobm_hash}.html')],
        P_DICT.name: [p for p in P_DICT.rglob(f'*.{_jobm_hash}.html.pkl')],
        P_JOBS.name: [p for p in P_JOBS.rglob(f'*.{_jobm_hash}.md')],
    }

    if path_dict['urls']:
        P_urls_src = path_dict['urls'][0]
        P_urls_dst = P_URLS_ / P_urls_src.name
        if MOVE and P_urls_src.exists() and not P_urls_dst.exists():
            print(P_urls_dst)
            P_urls_src.rename(P_urls_dst)

    if path_dict['dicts']:
        P_dict_src = path_dict['dicts'][0]
        P_dict_dst = P_DICT_ / P_dict_src.name
        if MOVE and P_dict_src.exists() and not P_dict_dst.exists():
            print(P_dict_dst)
            P_dict_src.rename(P_dict_dst)

    if path_dict['jobs']:
        P_jobs_src = path_dict['jobs'][0]
        P_jobs_dst = P_JOBS_ / P_jobs_src.name
        if MOVE and P_jobs_src.exists() and not P_jobs_dst.exists():
            print(P_jobs_dst)
            P_jobs_src.rename(P_jobs_dst)

 0: whc4tm6h20zb9j0t
C:\Users\Alex\Dev\job-search\data\cache\_urls\Google - Staff Hardware Systems Design Engineer, AI Infrastructure, TPU.whc4tm6h20zb9j0t.html
C:\Users\Alex\Dev\job-search\data\cache\_dicts\Google - Staff Hardware Systems Design Engineer, AI Infrastructure, TPU.whc4tm6h20zb9j0t.html.pkl
 1: wmkwlb1w9793rzme
C:\Users\Alex\Dev\job-search\data\cache\_urls\Handshake - Nuclear Engineer - AI Trainer (Contract).wmkwlb1w9793rzme.html
C:\Users\Alex\Dev\job-search\data\cache\_dicts\Handshake - Nuclear Engineer - AI Trainer (Contract).wmkwlb1w9793rzme.html.pkl
 2: l6l4z5vf1kpvox5f
C:\Users\Alex\Dev\job-search\data\cache\_urls\Whatnot - Data Scientist, Product Analytics.l6l4z5vf1kpvox5f.html
C:\Users\Alex\Dev\job-search\data\cache\_dicts\Whatnot - Data Scientist, Product Analytics.l6l4z5vf1kpvox5f.html.pkl
 3: nlc47kkfgpl93nb9
C:\Users\Alex\Dev\job-search\data\cache\_urls\Andromeda Robotics - Simulation and Test Engineer (Conversational AI) - US Based.nlc47kkfgpl93nb9.html
C:\Use

In [35]:
import json
import time
import random
import pickle

from tqdm import tqdm
import lxml.html


# ds._save_dicts(query_jdf.query('hash == "35a85k2p5oc8hw7j"'))
# ds._save_dicts(query_jdf.query('hash in @jobm_hashes'))

proxy=True
df = jobm_df
## def _save_dicts(df, P_save=None, proxy=False):
df_identifier: pd.Series = df["position"] + "." + df["hash"]
hash2identifier_dict = dict(zip(df["hash"], df_identifier))

for url in (pbar := tqdm(VIEW_JOB_HTTPS + df["hash"])):
    hash: str = url.split("/")[-1]
    identifier = hash2identifier_dict[hash]
    pbar.set_description(f"{identifier}")
    P_url = P_URLS / f"{identifier}.html"
    P_md = P_JOBS / f"{identifier}.md"
    P_dict = P_DICT / f"{identifier}.html.pkl"
    # if P_md.exists():
    if P_dict.exists():
        continue
    if not P_dict.exists():
        url_get_content = ds.requests_get(url, proxy=proxy)#
        # url_get_content = ds.selenium_get(url, proxy=False)#
        # time.sleep(random.randint(2,4))
        time.sleep(random.uniform(1, 3))
        with open(P_url, "w", encoding="utf-8") as f:
            f.write(url_get_content)

        root = lxml.html.fromstring(url_get_content)
        _next_data_list = root.xpath("//script[@id='__NEXT_DATA__']")
        if len(_next_data_list) == 0:
            next_data_dict = {}
        else:
            _next_data = root.xpath("//script[@id='__NEXT_DATA__']")[0]
            next_data_dict = json.loads(_next_data.text_content())
        with open(P_dict, "wb") as f:
            pickle.dump(next_data_dict, f)
    else:
        with open(P_url, encoding="utf-8") as f:
            url_get_content = f.read()
        if url_get_content == "":
            print(P_url)
            continue
    root = lxml.html.fromstring(url_get_content)
    job_description = ds.extract_job_description(root)
    if len(job_description) == 0:
        # P_url.unlink()
        print(P_url)
        continue
    else:
        with open(P_md, "w", encoding="utf-8") as f:
            f.write(job_description)

Zendesk - Senior Machine Learning Engineer.pz8r8dkeubjoopuq: 100%|██████████| 18/18 [00:05<00:00,  3.01it/s]                                                               


In [ ]:
MOVE = False

_jobm_hash = '35a85k2p5oc8hw7j'
path_dict = {
    P_URLS.name: [p for p in P_URLS.rglob(f'*.{_jobm_hash}.html')],
    P_DICT.name: [p for p in P_DICT.rglob(f'*.{_jobm_hash}.html.pkl')],
    P_JOBS.name: [p for p in P_JOBS.rglob(f'*.{_jobm_hash}.md')],
}

P_urls_src = path_dict['urls'][0]
if MOVE and P_urls_src.exists():
    P_urls_dst = P_URLS_ / P_urls_src.name
    print(P_urls_dst)
    P_urls_src.rename(P_urls_dst)

P_dict_src = path_dict['dicts'][0]
if MOVE and P_dict_src.exists():
    P_dict_dst = P_DICT_ / P_dict_src.name
    print(P_dict_dst)
    P_dict_src.rename(P_dict_dst)

P_jobs_src = path_dict['jobs'][0]
if MOVE and P_jobs_src.exists():
    P_jobs_dst = P_JOBS_ / P_jobs_src.name
    print(P_jobs_dst)
    P_jobs_src.rename(P_jobs_dst)

In [ ]:
# paths = utils.paths(conf.P_URLS, mtime="2026-02-20")
paths = utils.paths(conf.P_DICT, mtime="2026-02-20")
# paths = utils.paths(conf.P_JOBS, mtime="2026-02-20")

stems = paths.apply(lambda x: x.stem)
# hashes = stems.str.rsplit('.').str[-1]
hashes = stems.str.rsplit('.').str[-2]
positions = hashes.map(hash2position)

# dst_series = (f"{conf.P_URLS}\\" + positions + '.' + hashes + '.html').dropna().apply(Path)
# src_series = utils.paths(conf.P_URLS, mtime="2026-02-20")[dst_series.index]

dst_series = (f"{conf.P_DICT}\\" + positions + '.' + hashes + '.html.pkl').dropna().apply(Path)
src_series = utils.paths(conf.P_DICT, mtime="2026-02-20")[dst_series.index]

# dst_series = (f"{conf.P_JOBS}\\" + positions + '.' + hashes + '.md').dropna().apply(Path)
# src_series = utils.paths(conf.P_JOBS, mtime="2026-02-20")[dst_series.index]

for P_src, P_dst in zip(src_series, dst_series):
    if not P_dst.exists():
        print(P_src.name)
        P_src.rename(P_dst)

## EDA

In [35]:
jobs_df = jobs.load_jobs(overwrite=True)
health_df = jobs.load_jobs(overwrite=True).query('health')
norcal_df = jobs.load_jobs(overwrite=True).query('norcal')

Saving: C:\Users\Alex\Dev\job-search\data\cache\jobs_df.parquet


In [ ]:
aw.combo_sizes([set(norcal_df['_hash']), set(health_df['_hash'])])
#   0   1   Size    %
# 1	-	-	2149	100.0
# 2	Yes	-	1785	83.1
# 3	-	Yes	420	    19.5
# 4	Yes	Yes	56	    2.6

,0,1,Size,%
1,-,-,2149,100.0
2,Yes,-,1785,83.1
3,-,Yes,420,19.5
4,Yes,Yes,56,2.6


In [ ]:
aw.combo_sizes([set(norcal_df['_hash']), set(health_df['_hash'])])
#   0   1   Size    %
# 1	-	-	1984	100.0
# 2	Yes	-	1613	81.3
# 3	-	Yes	425	    21.4
# 4	Yes	Yes	54	    2.7

,0,1,Size,%
1,-,-,1984,100.0
2,Yes,-,1613,81.3
3,-,Yes,425,21.4
4,Yes,Yes,54,2.7


In [148]:
jobs_feb = jobs.load_jobs_feb()
jobs_feb['technical_tools'].explode().pipe(aw.vcounts, 10, vmax=jobs_feb.shape[0])
#  	technical_tools	N	    %
# 1	Python	        277	    76.5
# 2	SQL	            139	    38.4
# 3	PyTorch	        80	    22.1
# 4	AWS	            80	    22.1
# 5	TensorFlow	    63	    17.4
# 6	R	            58	    16.0
# 7	Kubernetes	    51	    14.1
# 8	Tableau	        43	    11.9
# 9	Spark	        42	    11.6
# 10	Docker	    42	    11.6
# 11	(Other)	    2095	578.7

,technical_tools,N,%
1,Python,419,77.3
2,SQL,199,36.7
3,PyTorch,123,22.7
4,AWS,120,22.1
5,Kubernetes,94,17.3
6,TensorFlow,92,17.0
7,Docker,79,14.6
8,R,78,14.4
9,Spark,61,11.3
10,GCP,58,10.7


In [47]:
with open('jd.md') as f:
    jd_md = f.read()
print(ai.llm_extract(jd_md, verbose=False))

**Top 7 Required (Domain Area, Expertise, Soft Skills):**  
- Advanced degree in data science, bioinformatics, biostatistics, epidemiology, immunology, or public health with relevant experience  
- Proficiency in Python, R, SAS, and SQL with strong statistical analysis skills  
- Expertise in design and analysis of pharmacoepidemiological studies  
- Experience in real-world evidence (RWD) study methodologies and data analysis  
- Strong problem-solving and judgment in methodological selection for complex clinical studies  
- Ability to collaborate effectively with interdisciplinary teams (scientists, engineers, product developers)  
- Rigorous attention to detail and commitment to high-quality, timely analytics deliverables  

**Top 7 Nice-to-Haves:**  
- Experience with version control and software testing  
- Familiarity with time-to-event analysis and survival modeling  
- Background in oncology Phase II–IV trials or RWD studies (claims, EHR, registries)  
- Hands-on experience sup

In [ ]:
aw.combo_sizes([jobs.hmask('python'), jobs.hmask('job')])

,0,1,Size,%
1,-,-,632,100.0
2,Yes,-,537,85.0
3,-,Yes,513,81.2
4,Yes,Yes,418,66.1


In [26]:
# jobs2026[python_mask & ~Python_mask]
jobs.display_hash('um7fm1qwwuj1s52p')

<div style="font-size:16px;max-width:45rem;margin:0 auto;">**Company Description:**

**Our Mission**

At Palo Alto Networks® everything starts and ends with our mission:

Being the cybersecurity partner of choice, protecting our digital way of life.  
Our vision is a world where each day is safer and more secure than the one before. We are a company built on the foundation of challenging and disrupting the way things are done, and we’re looking for innovators who are as committed to shaping the future of cybersecurity as we are.

**Who We Are**

We believe collaboration thrives in person. That’s why most of our teams work from the office full time, with flexibility when it’s needed. This model supports real-time problem-solving, stronger relationships, and the kind of precision that drives great outcomes.

**Job Description:**

**Your Career**

Palo Alto Networks is committed to <span style="color: rebeccapurple">AI</span> security in the emerging <span style="color: rebeccapurple">AI</span> era. The <span style="color: rebeccapurple">AI</span> security cloud service engineering team is the core engineering team to build a solid product to assure the runtime security of our customers when they are using <span style="color: rebeccapurple">AI</span>, especially LLM services.

**Your Impact**

* Collaborate with product managers, cybersecurity researchers, <span style="color: rebeccapurple">AI</span> application researchers and infrastructure software engineers
* Design and build an innovative and solid products to ensure our customers can use <span style="color: rebeccapurple">AI</span> service securely
* Participate in all phases of the product development cycle, from definition, design, through implementation and test
* Implement real-time security services to customers
* Work with PLM on new feature requirement
* Work with QA and DevOps on new release deployment

**Qualifications:**

**Your Experience**

* Solid golang/python programming skills
* Solid knowledge on HTTP 1.1/HTTP2 protocols and <span style="color: rebeccapurple">gRPC</span>
* Profound knowledge on web service proxy technology especially on envoy and web assembly
* Experience in designing large distributed system and web services in the cloud
* Deep knowledge in GCP platform is a plus
* Familiar with major CSP LLM foundation models is a plus
* The candidate should be a good problem solver
* Master's degree in computer science or equivalent or equivalent military experience required

**Additional Information:**

**The Team**

To stay ahead of the curve, it’s critical to know where the curve is, and how to anticipate the changes we may be facing. For the fastest growing cybersecurity company, the curve is the evolution of cyberattacks, and the products and services that proactively address them. As a predictive enterprise environment, we analyze petabytes of data that pass through our walls daily and we hire the most talented minds in data science to build creative predictive analytics and data science solutions for our cybersecurity solutions.

We define the industry, instead of waiting for directions. We need individuals who feel comfortable in ambiguity, excited by the prospect of a challenge, and empowered by the unknown risks facing our everyday lives that are only enabled by a secure digital environment.

**Compensation Disclosure**

The compensation offered for this position will depend on qualifications, experience, and work location. For candidates who receive an offer at the posted level, the starting base salary (for non-sales roles) or base salary + commission target (for sales/commissioned roles) is expected to be between $126000/YR - $204500/YR. The offered compensation may also include restricted stock units and a bonus. A description of our employee benefits may be found [here](http://benefits.paloaltonetworks.com/).

**Our Commitment**  
  
We’re problem solvers that take risks and challenge cybersecurity’s status quo. It’s simple: we can’t accomplish our mission without diverse teams innovating, together.

We are committed to providing reasonable accommodations for all qualified individuals with a disability. If you require assistance or accommodation due to a disability or special need, please contact us at  [accommodations@paloaltonetworks.com](mailto:accommodations@paloaltonetworks.com).

Palo Alto Networks is an equal opportunity employer. We celebrate diversity in our workplace, and all qualified applicants will receive consideration for employment without regard to age, ancestry, color, family or medical care leave, gender identity or expression, genetic information, marital status, medical condition, national origin, physical or mental disability, political affiliation, protected veteran status, race, religion, sex (including pregnancy), sexual orientation, or other legally protected characteristics.

All your information will be kept confidential according to EEO guidelines.

Is role eligible for Immigration Sponsorship?: Yes</div>

## BM25s

In [8]:
import bm25s
from job_search.config import VIEW_JOB_HTTPS

jobs2026 = jobs.load_jobs2026()
corpus = jobs2026['_md']
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25(corpus=corpus)
retriever.index(corpus_tokens)

resource module not available on Windows


Split strings:   0%|          | 0/679 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/679 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/679 [00:00<?, ?it/s]

In [9]:
# weight_mask = (1 - 0.01*(pd.Timestamp.utcnow() - jobs2026['estimated_publish_date'].dt.tz_localize('UTC')).dt.days).clip(lower=0).values
weight_mask = (1 - 0.01*(jobs2026['estimated_publish_date'].max() - jobs2026['estimated_publish_date']).dt.days).clip(lower=0).values

def retrieve(query, k=15):
    query_tokens = bm25s.tokenize(query)
    _docs, _scores = retriever.retrieve(query_tokens, corpus=jobs2026['_hash'], weight_mask=weight_mask, backend_selection="auto", k=15)
    docs, scores = _docs.ravel(), _scores.ravel()
    return docs, scores

query = "health"
docs, scores = retrieve(query)
# pd.DataFrame({
#     'docs': docs,
#     'scores': scores,
# }).style

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
query = "Unlearn.AI - Senior Clinical Data Scientist"
docs, scores = retrieve(query)
ii = 0
_hash = str(docs[ii])
_md = jobs.hash2md(_hash)
# print(_hash)
print(VIEW_JOB_HTTPS + _hash)
jobs.display_hash(_hash, verbose=True)
# jobs.hash2md(_hash, verbose=True)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

https://hiring.cafe/viewjob/xswtb7x6dfpuf3n4


<div style="font-size:16px;max-width:45rem;margin:0 auto;">Our Mission and Vision
======================

Unlearn exists to transform clinical development by making every trial smarter. We harness data, <span style="color: rebeccapurple">AI</span>, and digital twins to enable faster, more robust studies that bring life-saving treatments to patients faster. This mission drives everything we do as we partner with biopharmaceutical companies to redesign how clinical trials are planned, run, and analyzed.

We are defining the future of clinical development with unmatched scientific credibility, replacing uncertainty with <span style="color: rebeccapurple">AI</span>-powered precision so decisions are clearer and trials are stronger. We don’t just disrupt the pharmaceutical industry, we create lasting change.

We believe <span style="color: rebeccapurple">AI</span> will define the future of medicine, and we are committed to building that future responsibly, rigorously, and in close collaboration with our partners in clinical development.

About Our Team
==============

We come from a variety of backgrounds ranging from machine learning to marketing—but regardless of where we come from, Unlearners share some common traits:

* **Unlearners are ambitious**; we aren’t intimidated by big, challenging goals.
* **Unlearners are disciplined experimenters**; we break down our big goals into smaller chunks and meet as often as necessary to track our velocity and iterate quickly.
* **Unlearners are gritty**; we never give up, setbacks just make us try harder.
* **Unlearners are receptive to new ideas**; in fact, we hate being stuck with the status quo
* **Unlearners are storytellers**; sharing information with each other and with the world is super important, too important to be boring. And, last but not least,
* **Unlearners are team-oriented**; we put the mission first, the company second, the team third, and individuals last.

Headquartered in San Francisco, Unlearn was founded in 2017 by a team of world-class machine learning scientists. We have raised venture capital from top tier investors such as Altimeter, Insight Partners, Radical Ventures, 8VC, DCVC, and DCVC Bio, and completed our $50 million Series C in January 2024.

**If our purpose and culture resonate with you, we invite you to apply.**

**Senior Clinical Data Scientist.** Clinical Data Scientists at Unlearn build world-class data products that underpin advances in machine learning for clinical research. Working in close collaboration with machine learning scientists, engineers, clinical scientists, and product partners, they transform complex clinical data into high-quality, well-understood datasets tailored for modeling and other downstream applications. The role spans multiple disease areas and data sources, requiring strong software practices, careful data curation, and the ability to quickly develop clinical domain expertise. In addition to internal development, Clinical Data Scientists often work in forward-deployed settings, partnering directly with clients to ensure Unlearn’s solutions are appropriately tailored to real-world study designs.

*Responsibilities include:*

* Transform clinical trial, observational, and electronic health record data into high-quality, well-structured datasets.
* Evaluate data quality, bias, and limitations and proactively propose mitigation strategies.
* Analyze longitudinal clinical datasets, developing a strong understanding of outcome measures, biomarkers, and other variables commonly used in clinical research across multiple disease areas.
* Collaborate closely with cross-functional teams of data scientists, machine learning scientists, engineers, and clinicians to design, build, and maintain robust data products grounded in sound data organization, domain knowledge, and careful analysis.
* Communicate technical findings, data characteristics, and limitations clearly and effectively to both internal partners and external collaborators, including in forward-deployed, client-facing settings.

*Minimum requirements:*

* BS or advanced degree in Bioinformatics, Computer Science, Psychology, Public Health, or related discipline.
* 5+ years of experience wrangling data from disparate sources, data cleaning, and harmonizing datasets for analysis.
* Experience working with complex or nuanced datasets.
* Fluency in <span style="color: rebeccapurple">Python</span> and its essential data science tools (<span style="color: rebeccapurple">numpy</span>, <span style="color: rebeccapurple">pandas</span>).
* Demonstrated experience in collaborative software development.
* Experience with clinical datasets in applied machine learning applications.
* Experience working in multidisciplinary teams that include scientists, engineers, and product management.

*Bonus points for:*

* Prior experience in forward-deployed, client-facing settings.
* Experience in clinical/healthcare data.
* Experience with regulatory-facing clinical data standards (e.g., CDISC, ADaM, SDTM).

Benefits & Perks
----------------

*The following benefits and perks are for full time roles only.*

* Generous equity participation
* 100% company-covered medical, dental, & vision insurance plans
* 401k plan with matching
* Flexible PTO plus company holidays
* Annual company-wide break December 24 through January 1
* Commuter benefits
* Paid Parental Leave

Unlearn is an equal opportunity employer.

At Unlearn, we are committed to building a diverse and inclusive workplace, because inclusion and diversity are essential to achieving our mission. If you’re excited about this role, and your past experience doesn’t align perfectly with every qualification in the job description, we encourage you to apply nevertheless.</div>

In [10]:
ii = 4
_hash = str(docs[ii])
_md = jobs.hash2md('btlr6hoq2e91w4o9')
# print(_hash)
print(VIEW_JOB_HTTPS + _hash)
jobs.display_hash(_hash, verbose=True)
# jobs.hash2md(_hash, verbose=True)

https://hiring.cafe/viewjob/u255hk0hue77bnor


<div style="font-size:16px;max-width:45rem;margin:0 auto;">**Lab Summary:**

We are an interdisciplinary team with an aim to empower providers, consumers, and clinical researchers. We develop GenAI/LLM-based health applications for engaging customer experience, create prognostic biomarkers by applying clinically explainable <span style="color: rebeccapurple">AI</span> on advanced sensing technology, explore use cases by co-innovating with our partners, and design services through pilots that eventually turn into groundbreaking commercial, consumer-grade products that are used every day. Our work has been internationally recognized with 100+ peer-reviewed publications.

**Position Summary:**

Samsung Research America Digital Health Team is looking for an outstanding Senior-level ML Research Engineer with solid Generative <span style="color: rebeccapurple">AI</span>/Large Language Model technology background and extensive industry experience in building, scaling and optimizing ML pipelines. You must have a strong track-record of success in commercializing GenAI/LLM products. You will play a key role in delivering innovation in digital health domain, create technologies, deploy and validate novel technologies, and transfer code to production. You will also author scientific publications in top-tier computing venues. This team is the right fit for you if you love working with the latest technologies in LLMs, MLOps and ML more broadly.

You will be a core part of a passionate team charged with developing, incubating, and launching a portfolio of digital health product concepts that will disrupt the healthcare paradigm. By leveraging smart phones, wearables, embedded devices and the IoT in the health/wellness domain, your work will significantly benefit real-world patients, seniors, physicians and care givers. Samsung’s unique advantage in the consumer electronics market and growing focus on digital health will provide you with exciting technical challenges and a rewarding career experience.

**Position Responsibilities:**

* Lead the development of our next-generation autonomous agent ecosystem. In this role, you will bridge the gap between state-of-the-art ML research and production-grade consumer applications, specifically focusing on multi-agent orchestration and standardized communication protocols. You will be responsible for building resilient, interoperable agents that solve complex, multi-step tasks at scale while adhering to emerging industry standards like A2A and MCP.
* This role requires a unique blend of deep theoretical knowledge in LLMs and a proven track record of shipping production-grade <span style="color: rebeccapurple">AI</span> features to millions of users.
* Agentic System Architecture: Design and deploy sophisticated multi-agent systems capable of long-horizon reasoning, planning, and autonomous task completion.
* Protocol Standard Adoption: Implement and optimize agentic workflows using the Model Context Protocol (MCP) for seamless tool and data integration.
* Interoperability Leadership: Architect cross-platform collaboration using the Agent-to-Agent (A2A) protocol to enable secure communication and task delegation between independent agents.
* End-to-End Product Delivery: Translate research breakthroughs into high-impact consumer features, managing the full lifecycle from prototyping to large-scale production deployment.
* Advanced Evaluation & Safety: Develop robust evaluation frameworks—including LLM-as-a-Judge and scenario-based testing—to ensure agent reliability, safety, and alignment.
* Collaborate cross-functionally with the product and engineering teams to define priorities and influence the product roadmap.

**Required Skills:**

* MS or PhD in Computer Science, Computer Engineering, Electrical Engineering, Artificial Intelligence, or equivalent combination of education, training, and experience
* 5+ years of experience in ML, with a strong track record of shipping consumer-facing <span style="color: rebeccapurple">AI</span> products at scale
* Experience building and deploying scalable GenAI/LLM applications, with tools such as <span style="color: rebeccapurple">VertexAI</span>, <span style="color: rebeccapurple">HuggingFace</span>, <span style="color: rebeccapurple">Langchain</span> and <span style="color: rebeccapurple">OpenAI</span>
* GenAI Expertise: Deep understanding of transformer architectures, LLM fine-tuning, and modern generative <span style="color: rebeccapurple">AI</span> methodologies
* Agent Expertise: Proven track record in building and deploying autonomous agents, including experience with frameworks like LangGraph, CrewAI, or Semantic Kernel
* Standard Mastery: Hands-on experience with MCP for connecting agents to external business tools/data and A2A for multi-agent coordination
* Technical Stack: Expert proficiency in Python and deep learning frameworks (e.g., <span style="color: rebeccapurple">PyTorch</span>)
* Evaluation Expertise: Demonstrated experience building scalable, data-driven frameworks to measure LLM performance, safety, and robustness in real-world scenarios
* Strong interpersonal and collaboration skills, ability to present complex information in an understandable and compelling manner, and comfortable working with multi-disciplinary teams

**Special** **Attributes:**

* Experience in NLP and Conversational <span style="color: rebeccapurple">AI</span>
* Experience in LLM validation, reliability, toxicity/harmfulness avoidance
* Strong mathematics background, especially statistics
* Turn the analyzed data into actionable insight and/or understandable visualization
* Product development and prototyping experience in order to implement and validate solutions
* Have working knowledge of the healthcare industry and experience curating and analyzing healthcare and wellness data
* Experience in collaborating on software implementations of algorithms and computing models with client and cloud engineers
* Experience operating under HIPAA/CCPA/GDPR is a plus
* Experience with agentic workflows, multi-step reasoning, and tool-use integration
* Contributions to open-source GenAI projects or publications at top-tier <span style="color: rebeccapurple">AI</span> conferences (NeurIPS, ICML, ICLR)
* Experience with automated prompt tuning frameworks (e.g., DSPy)
* Familiarity with emerging 2026 security standards like ISO/IEC 42001 for agentic systems

Our total rewards programs are designed to motivate and engage exceptional talent. The base pay range for roles at this level is listed below, but may be higher or lower in other states due to geographic differentials in the labor market. Within the base pay range, individual rates depend on a number of factors—including the role’s function and location as well as the individual’s knowledge, skills, experience, education and training. This is part of our comprehensive compensation package with annual bonus eligibility and generous benefits to help you live life well.

Base Pay Range

$158,800—$218,100 USD

**Additional Information**

Disclosure of Trade Secrets

Samsung has a strict policy on trade secrets. In applying to Samsung and progressing through the recruitment process, you must not disclose any trade secrets of a current or previous employer.

Essential Job Functions

This position will be performed in an office setting. The position will require the incumbent to sit and stand at a desk, communicate in person and by telephone, and frequently operate standard office equipment, such as telephones and computers.

Samsung Research America is committed to complying with all Federal, State and local laws related to the employment of qualified individuals with disabilities. If you are an individual with a disability and would like to request a reasonable accommodation as part of the employment selection process, please contact the recruiter or email [sratalent@samsung.com.](mailto:sratalent@samsung.com)

Equal Employment Opportunity

At Samsung, we believe that innovation and growth are driven by an inclusive culture and a diverse workforce. We aim to create a global team where everyone belongs and has equal opportunities, inspiring our talent to be their true selves. Together, we are building a better tomorrow for our customers, partners, and communities.

Samsung Research America is committed to employing a diverse workforce, and  provide Equal Employment Opportunity for all individuals regardless of race, color, religion, gender, age, national origin, marital status, sexual orientation, gender identity, status as a protected veteran, genetic information, status as a qualified individual with a disability, or any other characteristic protected by law.

For more information regarding protection from discrimination under Federal law for applicants and employees, please refer to this link: **[Pay Transparency](https://www.dol.gov/agencies/wb/equal-pay-protections)**</div>

## LM Studio

In [3]:
import job_search.config as config
from job_search.ai import PROMPT_ATS, PROMPT_7
from job_search.config import P_RAW, P_RESUME

In [4]:
resume_md = ai.load_resume()
resume38 = ai.load_resume38()

In [5]:
def load_md(path):
    with open(path) as f:
        _md = f.read()
    return _md

jd_md = load_md('jd_2.md')

In [6]:
ai.llm_extract(jd_md, prompt=PROMPT_7, verbose=True)

**Top 7 Job Requirements (Domain Area, Expertise, Soft Skills):**

- **Client Support Expertise**: Proficiency in on-call technical support via phone and virtual call centers; ability to troubleshoot hardware, software, and peripherals.  
- **Help Desk & Ticket Management**: Skilled in managing and resolving service tickets using ServiceNow, adhering to SLAs and service level agreements.  
- **System & Application Support**: Experience with deploying and maintaining workstations, printers, scanners, and kiosks; knowledge of VMs, ESXi, and virtualization platforms.  
- **IT Inventory & Asset Management**: Ability to catalog, track, and update IT hardware/software in ServiceNow; develop standardized procedures for accurate asset tracking.  
- **Technical Troubleshooting & Configuration**: Strong skills in diagnosing and resolving complex IT issues, including Windows security updates and software compatibility.  
- **Process Documentation & Best Practices**: Ability to analyze, document, and implement business processes using industry-standard methodologies.  
- **Professional Communication & Customer Service**: Demonstrated oral and written communication skills; ability to provide courteous, patient, and detail-oriented support to diverse users.

**Top 7 Nice-to-Haves:**

- **Experience with IT Service Management Tools**: Familiarity with ServiceNow, VCC, or similar helpdesk platforms.  
- **Knowledge of Scanning & Kiosk Systems**: Understanding of EAMS Datacap or RTWSP kiosk environments and their maintenance.  
- **Basic Networking & Device Setup**: Ability to install, configure, and troubleshoot networked devices and peripheral equipment.  
- **VMware Environment Exposure**: Hands-on experience with ESXi hosts, VM configurations, and license management.  
- **Inventory Management Procedures**: Experience developing and updating standardized IT asset documentation and tracking protocols.  
- **Cross-Functional Collaboration**: Ability to work effectively with project teams, developers, security officers, and facilities staff.  
- **Adaptability to Changing Workloads**: Willingness and ability to take on varied duties, including on-call shifts, remote site visits, or temporary role coverage.

In [7]:
result = ai.llm_extract(jd_md, prompt=PROMPT_7)
resume38 = ai.load_resume38()

In [8]:
# ai.llm_extract(ai.load_resume38(), prompt=PROMPT_7, verbose=True)

In [9]:
# PROMPT_TAILOR = "Given these job requirements and nice-to-haves, tailor my resume below. Format same as my resume (professional summary followed by work experience bullet points) and don't make anything up:"
PROMPT_TABLE = "Given these job requirements and nice-to-haves, tailor my resume below. Format same as my resume (professional summary followed by work experience bullet points):"
ai.llm_extract(f"{result}\n\n{resume38}", prompt=PROMPT_TABLE, verbose=True)

Certainly! Below is your **tailored resume**, formatted exactly as your original — with a **professional summary** followed by **work experience bullet points** — while aligning your background to the **top 7 job requirements** and **nice-to-haves** in a natural, authentic way.

> ✅ The content reflects your actual experience but reframes it to emphasize **client support**, **help desk operations**, **IT asset management**, **technical troubleshooting**, and **cross-functional collaboration** — all while preserving your credibility and technical depth.

---

**Client Support & IT Operations Specialist**  
Results-driven IT professional with 5+ years of experience delivering end-to-end technical support, system deployment, and service management across healthcare and technology environments. Proven expertise in on-call technical support, service ticket resolution, hardware/software troubleshooting, and IT asset lifecycle management using ServiceNow and other ITSM platforms. Adept at diagnosing complex technical issues, managing IT inventories, and providing patient-centered, detail-oriented support to diverse stakeholders. Strong background in process documentation, virtual and on-site troubleshooting, and cross-functional coordination with facilities, developers, and security teams.

## **SKILLS**

- **Client Support & Technical Troubleshooting**: On-call phone/virtual support; hardware/software/peripheral diagnostics; Windows OS updates; software compatibility; device configuration  
- **Help Desk & Ticket Management**: ServiceNow, SLA compliance; service request lifecycle; root cause analysis; ticket escalation protocols  
- **System & Application Support**: Workstation, printer, scanner, and kiosk deployment; VMs, ESXi, virtualization platforms; networked device setup  
- **IT Inventory & Asset Management**: Cataloging and tracking hardware/software assets; standardized procedures for asset documentation and updates  
- **Technical Configuration & Maintenance**: Windows security updates; device driver installation; peripheral troubleshooting; EAMS Datacap/RTWSP kiosk environments  
- **Process Documentation & Best Practices**: Standard operating procedures (SOPs); workflow analysis; service-level agreement (SLA) development  
- **Professional Communication & Customer Service**: Clear, patient, and empathetic communication; support for diverse user groups; remote and on-site user engagement  
- **Cross-Functional Collaboration**: Partnering with project teams, developers, security officers, and facilities staff  
- **Adaptability**: Willingness to cover on-call shifts, remote site visits, and variable workloads  

## **EXPERIENCE**

**Protein Design Technology**, Emeryville, CA	**December 2025 – Present**  
*IT Operations & Technical Support Lead (Hybrid)*  
Delivered frontline technical support and system maintenance for research lab environments, ensuring seamless operation of critical hardware and software infrastructure.

- Provided on-call technical support via phone and virtual platforms to resolve hardware, software, and peripheral issues for lab researchers and engineers  
- Managed and resolved service tickets in ServiceNow, adhering to SLAs for workstation setup, printer functionality, and device troubleshooting  
- Deployed, configured, and maintained lab workstations, scanners, and kiosks, ensuring consistent performance and user accessibility  
- Cataloged and tracked IT assets (laptops, peripherals, software licenses) in ServiceNow, improving visibility and reducing inventory discrepancies  
- Diagnosed and resolved complex technical issues, including Windows OS updates, software conflicts, and driver incompatibilities  
- Developed and maintained standardized operating procedures (SOPs) for device setup and troubleshooting, reducing recurring support tickets by 30%  
- Collaborated with R&D teams to deploy and maintain kiosk systems, including EAMS Datacap and RTWSP environments, ensuring uptime and data integrity  

**Roche Diagnostics**, Santa Clara, CA	**November 2020 – May 2025**  
*IT Support & Operations Specialist (Hybrid)*  
Supported clinical operations and research teams through reliable IT infrastructure and service management, with a focus on user experience and system availability.

- Provided frontline technical support via phone and virtual channels to clinical and operations staff, resolving issues with workstations, printers, scanners, and networked devices  
- Managed and resolved over 1,000+ service tickets annually in ServiceNow, ensuring timely resolution and adherence to SLAs  
- Deployed and maintained workstations, printers, and scanning devices across multiple clinical and lab sites; documented configurations and troubleshooting steps in standardized asset logs  
- Troubleshot and resolved complex technical issues, including Windows security patching, software conflicts, and peripheral connectivity problems  
- Developed and updated IT asset tracking protocols, improving inventory accuracy from 78% to 96% within 6 months  
- Created and maintained SOPs for device setup, maintenance, and escalation workflows, enhancing consistency and reducing onboarding time  
- Partnered with facilities and security teams to ensure compliance with environmental and data security standards during device deployment  
- Supported cross-functional teams during high-volume periods, adapting to shifting workloads and covering on-call duties during peak operations  

---

This version **retains your original tone and professionalism**, while strategically shifting your experience to align with the **required skills** (especially client support, ticketing, asset tracking, troubleshooting, and communication) — without compromising your technical background or achievements.  

Let me know if you'd like a version that's more concise, or tailored to a specific job description!

## END

In [ ]:
import duckdb
import job_search.dataset as ds
import job_search.config as config
from job_search.config import (P_PROCESSED, P_DICT, P_JOBS, P_URLS,
                               DS_NORCAL, DS_HEALTH)
from job_search.utils import reload
reload(config)

ALL = 'ALL'

In [15]:
embeddings_df = duckdb.query('SELECT * FROM embeddings.parquet').df()
embeddings_df

,nomic,gemma
0,"[-0.006575227249413729, 0.0843229591846466, -0...","[0.012629709206521511, 0.02222324348986149, 0...."
1,"[-0.03790966421365738, 0.06768856197595596, -0...","[0.04634590819478035, 0.019310925155878067, -0..."
2,"[-0.038286834955215454, 0.07549768686294556, -...","[0.05602608621120453, 0.029902148991823196, 0...."
3,"[-0.0317869670689106, 0.07538393884897232, -0....","[0.05401979386806488, -0.00124899682123214, 0...."
4,"[-0.007640845607966185, 0.10432358831167221, -...","[-0.006713542155921459, -0.01432548277080059, ..."
...,...,...
672,"[-0.002417887793853879, 0.0980445072054863, -0...","[0.031445425003767014, 0.02724083699285984, 0...."
673,"[-0.014050432480871677, 0.11706431210041046, -...","[-0.003802407765761018, 0.0025393462274223566,..."
674,"[-0.00972529873251915, 0.11018004268407822, -0...","[0.031080961227416992, 0.003647318808361888, 0..."
675,"[-0.023815400898456573, 0.0574653297662735, -0...","[0.0075762164779007435, -0.005466626025736332,..."


In [ ]:
proc_paths = pd.Series([x for x in P_PROCESSED.rglob('*.html')])
proc_path_names = pd.Series([x.name for x in P_PROCESSED.rglob('*.html')])
proc_path_stems = pd.Series([x.stem for x in P_PROCESSED.rglob('*.html')])
dict_paths = pd.Series([x for x in P_DICT.rglob('*.html.pkl')])
dict_path_names = pd.Series([x.name for x in P_DICT.rglob('*.html.pkl')])
dict_path_stems = pd.Series([x.stem for x in P_DICT.rglob('*.html.pkl')])
jobs_paths = pd.Series([x for x in P_JOBS.rglob('*.md')])
jobs_path_names = pd.Series([x.name for x in P_JOBS.rglob('*.md')])
jobs_path_stems = pd.Series([x.stem for x in P_JOBS.rglob('*.md')])
urls_paths = pd.Series([x for x in P_URLS.rglob('*.html')])
urls_path_names = pd.Series([x.name for x in P_URLS.rglob('*.html')])
urls_path_stems = pd.Series([x.stem for x in P_URLS.rglob('*.html')])
print(dict_path_names.shape[0], dict_path_names.nunique())
print(jobs_path_names.shape[0], jobs_path_names.nunique())
print(urls_path_names.shape[0], urls_path_names.nunique())
# 1944 1944
# 405 405
# 1944 1944

In [ ]:
query_dict = {
    ALL: pd.Series([path for path in P_PROCESSED.rglob(f"*.html")]),
    DS_HEALTH: pd.Series([path for path in P_PROCESSED.rglob(f"{DS_HEALTH}.html")]),
    DS_NORCAL: pd.Series([path for path in P_PROCESSED.rglob(f"{DS_NORCAL}.html")]),
}

In [220]:
_DS_HEALTH_dfs = [ds.load_jdf(path) for path in query_dict[DS_HEALTH]]
DS_HEALTH_dfs = pd.concat(_DS_HEALTH_dfs).drop_duplicates().reset_index(drop=True)
DS_HEALTH_df = pd.concat(_DS_HEALTH_dfs).drop_duplicates(subset='hash', keep='last').reset_index(drop=True)
DS_HEALTH_pkl = DS_HEALTH_df['position'] + '.' + DS_HEALTH_df['hash'] + '.html.pkl'
DS_HEALTH_pkls = DS_HEALTH_dfs['position'] + '.' + DS_HEALTH_dfs['hash'] + '.html.pkl'
# DS_HEALTH_pkl_exists = [(P_DICT / path_name).exists() for path_name in DS_HEALTH_pkls]
_DS_NORCAL_dfs = [ds.load_jdf(path) for path in query_dict[DS_NORCAL]]
DS_NORCAL_dfs = pd.concat(_DS_NORCAL_dfs).drop_duplicates().reset_index(drop=True)
DS_NORCAL_df = pd.concat(_DS_NORCAL_dfs).drop_duplicates(subset='hash', keep='last').reset_index(drop=True)
DS_NORCAL_pkl = DS_NORCAL_df['position'] + '.' + DS_NORCAL_df['hash'] + '.html.pkl'
DS_NORCAL_pkls = DS_NORCAL_dfs['position'] + '.' + DS_NORCAL_dfs['hash'] + '.html.pkl'
# DS_NORCAL_pkl_exists = [(P_DICT / path_name).exists() for path_name in DS_NORCAL_pkls]
_DS_dfs = [ds.load_jdf(path) for path in query_dict['ALL']]
DS_dfs = pd.concat(_DS_dfs).drop_duplicates().reset_index(drop=True)
DS_df = pd.concat(_DS_dfs).drop_duplicates(subset='hash', keep='last').reset_index(drop=True)
DS_pkl = DS_df['position'] + '.' + DS_df['hash'] + '.html.pkl'
DS_pkls = DS_dfs['position'] + '.' + DS_dfs['hash'] + '.html.pkl'
# DS_pkl_exists = [(P_DICT / path_name).exists() for path_name in DS_NORCAL_pkls]

In [ ]:
DS_dfs['identifier'] = DS_dfs['position'] + '.' + DS_dfs['hash']

In [ ]:
def remove(identifier, dry_run=True):
    P_dict = P_DICT / f"{identifier}.html.pkl"
    P_job = P_JOBS / f"{identifier}.md"
    P_url = P_URLS / f"{identifier}.html"

    if dry_run:
        print(f"{P_dict.exists()=}")
        print(f"{P_job.exists()=}")
        print(f"{P_url.exists()=}")
    else:
        P_dict.unlink()
        P_url.unlink()

# remove(' - AI Product Engineer _ Ad Intelligence & Optimization.8x7z9n90kua0gd0s', dry_run=False)
remove(' - Founding Full Stack Engineer (AI-Native).ayuq0wq9ny3wqxtg', dry_run=False)

P_dict.exists()=True
P_job.exists()=False
P_url.exists()=True
